In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures that .env is loaded even if the env vars are already set

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures that .env is loaded even if the env vars are already set

token = os.environ.get("NEO4J_PASSWORD")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"NEO4J_PASSWORD loaded ({len(token)} characters): {masked}")
else:
    print("NEO4J_PASSWORD not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'NEO4J_PASSWORD', and load_dotenv() ran without error.")

NEO4J_PASSWORD loaded (43 characters): 8QGO...EuxY


In [4]:
"""
Research-gap analysis over your Neo4j knowledge graph.

Different from rag_ask.py: that script answers "what did the literature
find" by handing the LLM raw source sentences, one per finding, because the
sentence IS the finding. This script answers "where is research thin or
missing," which is a counting question, not a reading-comprehension one --
handing an LLM hundreds of sentences and asking it to judge what's common
vs. rare means it's silently tallying frequency from prose, which it does
unreliably past a couple dozen items. Instead, this script lets Neo4j do
the counting (exact, via COUNT/GROUP BY) and hands the LLM a small table of
already-correct numbers to reason about.

Three things get pulled and handed to the model:
  A) how many distinct entities exist per taxonomy label (axis size)
  B) for every label pair that's ever connected, what fraction of the
     source label's entities are ever connected to the target label at all
     (a coverage percentage -- low numbers are the actual gap signal)
  C) the individually thinnest- and best-studied entities, by relationship
     count and distinct-DOI count, as concrete named examples

Works against Aura or Desktop identically -- point NEO4J_URI at whichever.

Designed for Jupyter/Colab execution. No __main__ guard -- just set the
CONFIG values below and run the whole cell/file.

pip install neo4j openai --break-system-packages
"""

import os

from neo4j import GraphDatabase
from openai import OpenAI

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
NEO4J_URI = "neo4j+s://f1cebb95.databases.neo4j.io"     # <- your Aura instance URI
NEO4J_USERNAME = "f1cebb95"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")   # <- fill in, or set the env var
NEO4J_DATABASE = "f1cebb95"

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")   # <- fill in, or set the env var
OPENAI_MODEL = "gpt-4o"
TOP_N_ENTITIES = 40   # how many best- and worst-studied individual entities to surface

QUESTION = (
    "Using only the data provided, identify where the real research gaps lie "
    "in this plant protein dataset."
)

REQUIREMENTS = """\
The data spans three axes: how the protein was obtained (extraction method),
what was done to it afterward (modification method), and what was measured
as a result (functional, physicochemical, structural, rheological, or
sensory property), applied across different source materials. Don't assume
which axis the gap is in, determine that from the data itself.

Use the occurrence counts and distinct-DOI counts provided as your evidence
of research volume, not whether a claim exists. Name at least 4-5 specific,
substantive examples per section below, not just the single most extreme
row in a table -- a table's most extreme value is sometimes a genuine
finding and sometimes just a one-off oddity with a single supporting row;
prefer patterns backed by multiple entities or a meaningful entity count
over a single data point, unless nothing more substantial exists.

Report:
1. Well-studied baseline: the label pairings and individual entities with
   the most supporting relationships and papers.
2. Thin coverage: specific entities or label pairings backed by only one
   paper or a handful of relationships, favoring cases involving several
   entities at similarly low coverage over an isolated single-row anomaly.
3. Plausible but under-tested: label pairings with a low coverage
   percentage in TABLE 2 (few of the possible entities in one axis ever
   connect to another axis), which suggests combinations that remain
   largely untested. Say explicitly what the untested combination is, not
   just the percentage.
4. Which single axis, source material, extraction method, or modification
   method, shows the sparsest research depth overall. Base this specifically
   on TABLE 2B's avg_relationships_per_entity and total_relationships, not
   on entity count alone (Table 1) -- a large axis with low relationships-
   per-entity is understudied relative to its own diversity even if it
   isn't the smallest axis by entity count. State the actual numbers.

Cite the exact counts given for every claim (entity/label names, counts,
and coverage percentages), never sentence-level content, since this
question is about research volume, not individual findings.
"""

OUT_PATH = "Research_Gap_Analysis.md"

# ---------------------------------------------------------------
# CONNECT
# ---------------------------------------------------------------
if not NEO4J_PASSWORD:
    raise SystemExit("Set NEO4J_PASSWORD above or as an environment variable first.")
if not OPENAI_API_KEY:
    raise SystemExit("Set OPENAI_API_KEY above or as an environment variable first.")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to {NEO4J_URI}")


def query(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(cypher, **params)]


# ---------------------------------------------------------------
# A) Per-label entity counts -- axis size
# ---------------------------------------------------------------
label_counts = query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS entity_count
    ORDER BY entity_count DESC
""")
label_entity_totals = {r["label"]: r["entity_count"] for r in label_counts}

# CORE_CATEGORIES restricts everything below to rows that actually represent
# "a treatment/material was tested and a property outcome was reported"
# (category = treatment_property_change or combined_treatment_property_change,
# 8,305 of the 10,324 relationships). Everything else in the graph -- direct
# protein-vs-protein comparisons, treatment-vs-treatment comparisons, cultivar
# comparisons, and so on -- is a different kind of claim, not a "has this
# combination been tested" data point, and mixing them in produced noise: a
# handful of one-off property-to-property comparison edges were dominating
# the "thin coverage" section as if they were meaningful research gaps, when
# they were really just single direct-comparison rows, not evidence of an
# untested combination.
CORE_CATEGORIES = ["treatment_property_change", "combined_treatment_property_change"]
TREATMENT_LABELS = ["MaterialUsedDirectly", "ExtractionMethod", "ModificationMethod"]

# ---------------------------------------------------------------
# B) Per-label-pair coverage -- of all entities in the source label, how
# many ever connect to the target label, and with how much evidence
# ---------------------------------------------------------------
pair_coverage = query("""
    MATCH (s)-[r]->(t)
    WHERE r.category IN $categories
    WITH labels(s)[0] AS source_label, labels(t)[0] AS target_label,
         s.name AS source_name, r.doi AS doi
    RETURN source_label, target_label,
           count(DISTINCT source_name) AS distinct_source_entities,
           count(DISTINCT doi) AS distinct_dois,
           count(*) AS total_relationships
    ORDER BY total_relationships ASC
""", categories=CORE_CATEGORIES)
for row in pair_coverage:
    total = label_entity_totals.get(row["source_label"], 0)
    row["source_label_total_entities"] = total
    row["pct_source_covered"] = round(100 * row["distinct_source_entities"] / total, 1) if total else None

# ---------------------------------------------------------------
# B2) Per-axis total research volume -- this is what actually answers
# "which axis is sparsest overall." Entity count alone (Table 1) tells you
# how many distinct materials/methods exist, not how much they've been
# studied -- a small, well-studied axis and a large, thinly-studied one can
# have similar entity counts but very different research attention.
# Restricted to the three treatment-side axes, since those are the ones the
# question is actually about.
# ---------------------------------------------------------------
axis_volume = query("""
    MATCH (s)-[r]->()
    WHERE r.category IN $categories AND labels(s)[0] IN $treatment_labels
    WITH labels(s)[0] AS source_label, count(*) AS total_relationships,
         count(DISTINCT r.doi) AS distinct_dois, count(DISTINCT s.name) AS distinct_entities_used
    RETURN source_label, total_relationships, distinct_dois, distinct_entities_used
    ORDER BY total_relationships ASC
""", categories=CORE_CATEGORIES, treatment_labels=TREATMENT_LABELS)
for row in axis_volume:
    total = label_entity_totals.get(row["source_label"], 0)
    row["entities_in_label"] = total
    row["avg_relationships_per_entity"] = round(row["total_relationships"] / total, 1) if total else None

# ---------------------------------------------------------------
# C) Per-entity long tail -- individually thinnest and best-studied,
# same category restriction as above for consistency
# ---------------------------------------------------------------
entity_stats = query("""
    MATCH (n)-[r]-()
    WHERE r.category IN $categories
    WITH n, labels(n)[0] AS label, count(r) AS relationship_count,
         count(DISTINCT r.doi) AS distinct_dois
    RETURN n.name AS entity, label, relationship_count, distinct_dois
    ORDER BY relationship_count ASC
""", categories=CORE_CATEGORIES)
driver.close()

thinnest = entity_stats[:TOP_N_ENTITIES]
best_studied = sorted(entity_stats, key=lambda r: -r["relationship_count"])[:TOP_N_ENTITIES]
print(f"Entities: {len(entity_stats)} total, showing {TOP_N_ENTITIES} thinnest and {TOP_N_ENTITIES} best-studied "
      f"(not all {len(entity_stats)} -- the extremes are what matter for this question, the middle is noise)")

# ---------------------------------------------------------------
# FORMAT CONTEXT
# ---------------------------------------------------------------
def fmt_table(rows, columns):
    lines = [" | ".join(columns)]
    for r in rows:
        lines.append(" | ".join(str(r[c]) for c in columns))
    return "\n".join(lines)


context = f"""
TABLE 1 -- entity counts per label (axis size, ALL relationships, unfiltered):
{fmt_table(label_counts, ["label", "entity_count"])}

TABLE 2 -- coverage between label pairs (source_label -> target_label), restricted to
rows where a treatment/material was tested and a property outcome was reported
(excludes direct comparison-type relationships, which aren't "was this tested" data):
{fmt_table(pair_coverage, ["source_label", "target_label", "distinct_source_entities",
                            "source_label_total_entities", "pct_source_covered",
                            "distinct_dois", "total_relationships"])}

TABLE 2B -- total research volume per treatment axis (source material, extraction
method, modification method), same category restriction as Table 2. This is the
table to use for "which axis is sparsest overall" -- entity count (Table 1) measures
how many distinct things exist in an axis, not how much research attention they've
received; avg_relationships_per_entity is the per-entity research-depth signal:
{fmt_table(axis_volume, ["source_label", "distinct_entities_used", "entities_in_label",
                          "total_relationships", "distinct_dois", "avg_relationships_per_entity"])}

TABLE 3a -- {TOP_N_ENTITIES} thinnest individually-studied entities (fewest relationships):
{fmt_table(thinnest, ["entity", "label", "relationship_count", "distinct_dois"])}

TABLE 3b -- {TOP_N_ENTITIES} best-studied entities (most relationships), as a baseline:
{fmt_table(best_studied, ["entity", "label", "relationship_count", "distinct_dois"])}
"""

# ---------------------------------------------------------------
# ASK THE LLM
# ---------------------------------------------------------------
system_prompt = (
    "You are analyzing pre-aggregated coverage statistics computed directly from a "
    "plant-protein food-science knowledge graph built from published literature. "
    "Answer using ONLY the tables provided in the user message. Never draw on outside "
    "knowledge, training data, or anything not explicitly present in those tables, even "
    "if you believe it to be true."
)

user_prompt = f"""{QUESTION}

{REQUIREMENTS}

{context}
"""

client = OpenAI(api_key=OPENAI_API_KEY)
response = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0,
)

answer = response.choices[0].message.content
print("\n" + "=" * 70 + "\n")
print(answer)

with open(OUT_PATH, "w", encoding="utf-8") as f:
    f.write(f"# {QUESTION}\n\n{answer}\n")
print(f"\nSaved to {OUT_PATH}")

Connected to neo4j+s://f1cebb95.databases.neo4j.io
Entities: 2216 total, showing 40 thinnest and 40 best-studied (not all 2216 -- the extremes are what matter for this question, the middle is noise)


Based on the data provided, here is the analysis of research coverage in the plant protein dataset:

1. **Well-studied baseline:**
   - **Solubility (FunctionalProperty):** 649 relationships, 141 distinct DOIs.
   - **Water holding capacity (FunctionalProperty):** 585 relationships, 139 distinct DOIs.
   - **Ultrasonication (ModificationMethod):** 345 relationships, 21 distinct DOIs.
   - **Foaming capacity (FunctionalProperty):** 340 relationships, 101 distinct DOIs.
   - **Oil holding capacity (FunctionalProperty):** 316 relationships, 88 distinct DOIs.

2. **Thin coverage:**
   - **Foaming time (FunctionalProperty):** 1 relationship, 1 distinct DOI.
   - **High gluten content (PhysicochemicalProperty):** 1 relationship, 1 distinct DOI.
   - **Hydrophobic amino acid content (Physicochem